In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

print("Files available in Kaggle:")
print("=" * 50)

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import pandas as pd

# Dataset paths
safety_path = "/kaggle/input/datasets/sakshibhosale1904/construction-safety-risk-dataset/safety_risk/safety.csv"

injury_path = "/kaggle/input/datasets/sakshibhosale1904/construction-safety-risk-dataset/safety_risk/Injury_prediction.csv"

# Load datasets
safety_df = pd.read_csv(safety_path)
injury_df = pd.read_csv(injury_path)

print("✅ Datasets loaded successfully!")

print("\n" + "="*60)
print("SAFETY DATASET")
print("="*60)
print("Shape:", safety_df.shape)
print("Columns:", safety_df.columns.tolist())

print("\nFirst 5 rows:")
display(safety_df.head())

print("\n" + "="*60)
print("INJURY PREDICTION DATASET")
print("="*60)
print("Shape:", injury_df.shape)
print("Columns:", injury_df.columns.tolist())

print("\nFirst 5 rows:")
display(injury_df.head())

In [ ]:
# ============================================
# STEP 3 — DATA QUALITY CHECK
# ============================================

print("=" * 60)
print("SAFETY DATASET")
print("=" * 60)

print("\nMissing values:")
print(safety_df.isnull().sum())

print("\nDuplicate rows:", safety_df.duplicated().sum())

print("\nRisk Level distribution:")
print(safety_df["Risk_Level"].value_counts())

print("\nRisk Level percentage:")
print(
    (safety_df["Risk_Level"].value_counts(normalize=True) * 100)
    .round(2)
)


print("\n" + "=" * 60)
print("INJURY PREDICTION DATASET")
print("=" * 60)

print("\nMissing values:")
print(injury_df.isnull().sum())

print("\nDuplicate rows:", injury_df.duplicated().sum())

print("\nInjury distribution:")
print(injury_df["Injury"].value_counts(dropna=False))

print("\nInjury percentage:")
print(
    (injury_df["Injury"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
)

In [ ]:
# ============================================
# STEP 4 — CLEAN DATASETS
# ============================================

# Make copies so the original data remains unchanged
safety_clean = safety_df.copy()
injury_clean = injury_df.copy()


# --------------------------------------------
# SAFETY DATASET
# --------------------------------------------

print("SAFETY DATASET")
print("-" * 50)

print("Rows before cleaning:", len(safety_clean))

# Remove duplicate rows
safety_clean = safety_clean.drop_duplicates().reset_index(drop=True)

print("Rows after cleaning:", len(safety_clean))
print("Duplicates removed:", len(safety_df) - len(safety_clean))


# --------------------------------------------
# INJURY DATASET
# --------------------------------------------

print("\nINJURY DATASET")
print("-" * 50)

print("Rows before cleaning:", len(injury_clean))

# Remove duplicate rows
injury_clean = injury_clean.drop_duplicates().reset_index(drop=True)

print("Rows after cleaning:", len(injury_clean))
print("Duplicates removed:", len(injury_df) - len(injury_clean))


# --------------------------------------------
# FINAL CHECK
# --------------------------------------------

print("\n" + "=" * 60)
print("FINAL DATASET SIZES")
print("=" * 60)

print("Safety dataset :", safety_clean.shape)
print("Injury dataset :", injury_clean.shape)

print("\nRemaining duplicates:")
print("Safety:", safety_clean.duplicated().sum())
print("Injury:", injury_clean.duplicated().sum())

In [ ]:
# ============================================
# STEP 5 — PREPARE FEATURES
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ============================================================
# 1. SAFETY DATASET
# ============================================================

print("=" * 60)
print("SAFETY DATASET PREPARATION")
print("=" * 60)

# Remove Project_ID because it is only an identifier
X_safety = safety_clean.drop(columns=["Project_ID", "Risk_Level"])

# Target
y_safety = safety_clean["Risk_Level"]

# Convert Yes/No into 1/0
binary_columns = [
    "Helmet",
    "Vest",
    "Gloves",
    "Safety_Shoes",
    "Accident"
]

for col in binary_columns:
    X_safety[col] = X_safety[col].map({
        "Yes": 1,
        "No": 0
    })

print("\nSafety Features:")
print(X_safety.head())

print("\nSafety Target:")
print(y_safety.head())

print("\nSafety Feature Data Types:")
print(X_safety.dtypes)


# ============================================================
# 2. INJURY DATASET
# ============================================================

print("\n" + "=" * 60)
print("INJURY DATASET PREPARATION")
print("=" * 60)

# Features
X_injury = injury_clean.drop(columns=["Injury"])

# Target
y_injury = injury_clean["Injury"]

# Convert categorical features into numerical columns
X_injury = pd.get_dummies(
    X_injury,
    drop_first=False,
    dtype=int
)

print("\nNumber of injury features after encoding:",
      X_injury.shape[1])

print("\nFirst 5 encoded rows:")
display(X_injury.head())

print("\nInjury Target:")
print(y_injury.head())

print("\nInjury classes:")
print(y_injury.value_counts())

In [ ]:
# ============================================
# STEP 6 — TRAIN / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

# ============================================================
# 1. SAFETY DATASET
# ============================================================

X_safety_train, X_safety_test, y_safety_train, y_safety_test = train_test_split(
    X_safety,
    y_safety,
    test_size=0.20,
    random_state=42,
    stratify=y_safety
)

print("=" * 60)
print("SAFETY DATASET SPLIT")
print("=" * 60)

print("Training samples:", len(X_safety_train))
print("Testing samples :", len(X_safety_test))

print("\nTraining distribution:")
print(y_safety_train.value_counts())

print("\nTesting distribution:")
print(y_safety_test.value_counts())


# ============================================================
# 2. INJURY DATASET
# ============================================================

X_injury_train, X_injury_test, y_injury_train, y_injury_test = train_test_split(
    X_injury,
    y_injury,
    test_size=0.20,
    random_state=42,
    stratify=y_injury
)

print("\n" + "=" * 60)
print("INJURY DATASET SPLIT")
print("=" * 60)

print("Training samples:", len(X_injury_train))
print("Testing samples :", len(X_injury_test))

print("\nTraining distribution:")
print(y_injury_train.value_counts())

print("\nTesting distribution:")
print(y_injury_test.value_counts())

In [ ]:
# ============================================
# STEP 7 — SAFETY RISK MODEL COMPARISON
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------
# Define models
# ------------------------------------------------

safety_models = {

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        max_depth=5
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


# ------------------------------------------------
# Train and evaluate
# ------------------------------------------------

results = []

for name, model in safety_models.items():

    print(f"\nTraining {name}...")

    model.fit(
        X_safety_train,
        y_safety_train
    )

    predictions = model.predict(
        X_safety_test
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_safety_test,
            predictions
        ),
        "Precision": precision_score(
            y_safety_test,
            predictions,
            average="weighted",
            zero_division=0
        ),
        "Recall": recall_score(
            y_safety_test,
            predictions,
            average="weighted",
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_safety_test,
            predictions,
            average="weighted",
            zero_division=0
        )
    })


# ------------------------------------------------
# Results table
# ------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("SAFETY RISK MODEL COMPARISON")
print("=" * 70)

display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}"
    })
)

In [ ]:
# ============================================
# STEP 8 — DETAILED SAFETY MODEL EVALUATION
# ============================================

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

best_safety_models = {
    "Random Forest": safety_models["Random Forest"],
    "Gradient Boosting": safety_models["Gradient Boosting"],
    "Extra Trees": safety_models["Extra Trees"]
}

for name, model in best_safety_models.items():

    predictions = model.predict(X_safety_test)

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print("\nClassification Report:")
    print(
        classification_report(
            y_safety_test,
            predictions,
            zero_division=0
        )
    )

    print("Confusion Matrix:")
    print(
        confusion_matrix(
            y_safety_test,
            predictions
        )
    )

    # Confusion matrix visualization
    plt.figure(figsize=(6, 5))

    sns.heatmap(
        confusion_matrix(
            y_safety_test,
            predictions
        ),
        annot=True,
        fmt="d",
        xticklabels=model.classes_,
        yticklabels=model.classes_
    )

    plt.title(f"{name} - Safety Risk Confusion Matrix")
    plt.xlabel("Predicted Risk")
    plt.ylabel("Actual Risk")
    plt.show()

In [ ]:
# ============================================
# STEP 9 — 5-FOLD CROSS-VALIDATION
# ============================================

from sklearn.model_selection import StratifiedKFold, cross_validate

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, model in best_safety_models.items():

    scores = cross_validate(
        model,
        X_safety,
        y_safety,
        cv=cv,
        scoring=[
            "accuracy",
            "precision_weighted",
            "recall_weighted",
            "f1_weighted"
        ],
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "CV Precision": scores["test_precision_weighted"].mean(),
        "CV Recall": scores["test_recall_weighted"].mean(),
        "CV F1": scores["test_f1_weighted"].mean()
    })


# Create results table
cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="CV F1",
    ascending=False
).reset_index(drop=True)


print("=" * 80)
print("SAFETY RISK — 5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)

display(
    cv_results_df.style.format({
        "CV Accuracy": "{:.4f}",
        "Accuracy Std": "{:.4f}",
        "CV Precision": "{:.4f}",
        "CV Recall": "{:.4f}",
        "CV F1": "{:.4f}"
    })
)

In [ ]:
# ============================================
# STEP 10 — FINAL SAFETY RISK MODEL
# ============================================

from sklearn.ensemble import GradientBoostingClassifier
import joblib
import os

# Create final model
final_safety_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Train on the complete Safety dataset
final_safety_model.fit(
    X_safety,
    y_safety
)

print("✅ Final Safety Risk model trained successfully!")
print("Training records:", len(X_safety))
print("Features:", X_safety.columns.tolist())
print("Classes:", final_safety_model.classes_)

In [ ]:
# ============================================
# STEP 11 — SAVE FINAL SAFETY RISK MODEL
# ============================================

import joblib
import os
import json

model_dir = "/kaggle/working/safety_risk_model"

os.makedirs(model_dir, exist_ok=True)

# 1. Save trained model
joblib.dump(
    final_safety_model,
    f"{model_dir}/safety_risk_model.pkl"
)

# 2. Save feature names
joblib.dump(
    list(X_safety.columns),
    f"{model_dir}/safety_risk_features.pkl"
)

# 3. Save model metadata
metadata = {
    "model_name": "Gradient Boosting",
    "model_type": "GradientBoostingClassifier",
    "target": "Risk_Level",
    "features": list(X_safety.columns),
    "classes": list(final_safety_model.classes_),
    "training_records": len(X_safety),
    "random_state": 42
}

with open(
    f"{model_dir}/metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("✅ Model files saved successfully!")

print("\nFiles created:")
for file in os.listdir(model_dir):
    print(" -", file)

In [ ]:
# ============================================
# STEP 13 — TEST SAVED SAFETY RISK MODEL
# ============================================

import joblib
import pandas as pd

# Load saved model
model_path = "/kaggle/working/safety_risk_model/safety_risk_model.pkl"
features_path = "/kaggle/working/safety_risk_model/safety_risk_features.pkl"

model = joblib.load(model_path)
features = joblib.load(features_path)

# --------------------------------------------
# Test Case 1 — High Risk Scenario
# --------------------------------------------
test_case = pd.DataFrame([{
    "Helmet": 0,
    "Vest": 0,
    "Gloves": 0,
    "Safety_Shoes": 0,
    "Accident": 1
}])

# Make sure feature order is correct
test_case = test_case[features]

# Prediction
prediction = model.predict(test_case)[0]

# Probabilities
probabilities = model.predict_proba(test_case)[0]

# Create probability dictionary
probability_dict = dict(
    zip(model.classes_, probabilities)
)

print("=" * 60)
print("SAFETY RISK MODEL — TEST RESULT")
print("=" * 60)

print("\nInput:")
print(test_case)

print("\nPredicted Risk Level:")
print(prediction)

print("\nRisk Probabilities:")
for risk, probability in probability_dict.items():
    print(f"{risk}: {probability * 100:.2f}%")

print("\n" + "=" * 60)
print("✅ Prediction completed successfully!")
print("=" * 60)

In [ ]:
# ============================================
# STEP 14 — INJURY PREDICTION MODEL COMPARISON
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --------------------------------------------
# Define models
# --------------------------------------------

injury_models = {

    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# --------------------------------------------
# Train and evaluate
# --------------------------------------------

injury_results = []

for name, model in injury_models.items():

    model.fit(X_injury_train, y_injury_train)

    predictions = model.predict(X_injury_test)

    accuracy = accuracy_score(
        y_injury_test,
        predictions
    )

    precision = precision_score(
        y_injury_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_injury_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_injury_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    injury_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

# --------------------------------------------
# Display results
# --------------------------------------------

injury_results_df = pd.DataFrame(injury_results)

injury_results_df = injury_results_df.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("INJURY PREDICTION MODEL COMPARISON")
print("=" * 70)

display(
    injury_results_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}"
    })
)

print("\n✅ Injury prediction models trained successfully!")

In [ ]:
# ============================================
# STEP 15 — INJURY MODEL 3-FOLD CROSS-VALIDATION
# ============================================

from sklearn.model_selection import StratifiedKFold, cross_validate

# 3-fold CV because the smallest class has only 3 samples
cv_injury = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

cv_injury_results = []

for name, model in injury_models.items():

    scores = cross_validate(
        model,
        X_injury,
        y_injury,
        cv=cv_injury,
        scoring=[
            "accuracy",
            "precision_weighted",
            "recall_weighted",
            "f1_weighted"
        ],
        n_jobs=-1
    )

    cv_injury_results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "CV Precision": scores["test_precision_weighted"].mean(),
        "CV Recall": scores["test_recall_weighted"].mean(),
        "CV F1": scores["test_f1_weighted"].mean()
    })

# --------------------------------------------
# Results
# --------------------------------------------

cv_injury_df = pd.DataFrame(cv_injury_results)

cv_injury_df = cv_injury_df.sort_values(
    by="CV F1",
    ascending=False
).reset_index(drop=True)

print("=" * 80)
print("INJURY PREDICTION — 3-FOLD CROSS-VALIDATION")
print("=" * 80)

display(
    cv_injury_df.style.format({
        "CV Accuracy": "{:.4f}",
        "Accuracy Std": "{:.4f}",
        "CV Precision": "{:.4f}",
        "CV Recall": "{:.4f}",
        "CV F1": "{:.4f}"
    })
)

print("\n✅ 3-fold cross-validation completed!")

In [ ]:
# ============================================
# STEP 16 — FINAL INJURY PREDICTION MODEL v1.0
# ============================================

from sklearn.linear_model import LogisticRegression

# Create final model
final_injury_model = LogisticRegression(
    max_iter=3000,
    random_state=42
)

# Train on ALL cleaned injury records
final_injury_model.fit(
    X_injury,
    y_injury
)

print("=" * 70)
print("FINAL INJURY PREDICTION MODEL v1.0")
print("=" * 70)

print("\n✅ Final model trained successfully!")

print("\nModel:")
print("Logistic Regression")

print("\nTraining records:")
print(len(X_injury))

print("\nNumber of features:")
print(X_injury.shape[1])

print("\nClasses:")
for cls in final_injury_model.classes_:
    print(" -", cls)

print("\nFeature columns:")
print(X_injury.columns.tolist())

In [ ]:
# ============================================
# STEP 17 — SAVE INJURY PREDICTION MODEL v1.0
# ============================================

import joblib
import os
import json

model_dir = "/kaggle/working/injury_prediction_model"

os.makedirs(model_dir, exist_ok=True)

# 1. Save trained model
joblib.dump(
    final_injury_model,
    f"{model_dir}/injury_prediction_model.pkl"
)

# 2. Save feature names
joblib.dump(
    list(X_injury.columns),
    f"{model_dir}/injury_prediction_features.pkl"
)

# 3. Save metadata
metadata = {
    "model_name": "Injury_Prediction_Model_v1.0",
    "model_version": "v1.0",
    "model_type": "LogisticRegression",
    "target": "Injury",
    "features": list(X_injury.columns),
    "classes": list(final_injury_model.classes_),
    "training_records": len(X_injury),
    "number_of_features": X_injury.shape[1],
    "cross_validation": {
        "method": "3-Fold Stratified Cross-Validation",
        "cv_accuracy": 0.6211,
        "cv_accuracy_std": 0.0151,
        "cv_f1_weighted": 0.5886
    },
    "random_state": 42
}

with open(
    f"{model_dir}/metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("=" * 70)
print("INJURY PREDICTION MODEL — FILES SAVED")
print("=" * 70)

for file in os.listdir(model_dir):
    print(" -", file)

print("\n✅ Injury Prediction Model v1.0 saved successfully!")

In [ ]:
# ============================================
# STEP 18 — VERIFY SAVED INJURY MODEL
# ============================================

import joblib

model_path = "/kaggle/working/injury_prediction_model/injury_prediction_model.pkl"
features_path = "/kaggle/working/injury_prediction_model/injury_prediction_features.pkl"

# Load model
loaded_injury_model = joblib.load(model_path)

# Load features
loaded_injury_features = joblib.load(features_path)

print("=" * 70)
print("INJURY PREDICTION MODEL — VERIFICATION")
print("=" * 70)

print("\n✅ Model loaded successfully!")

print("\nModel type:")
print(type(loaded_injury_model).__name__)

print("\nNumber of features:")
print(len(loaded_injury_features))

print("\nClasses:")
for cls in loaded_injury_model.classes_:
    print(" -", cls)

print("\nFeatures loaded:")
print(loaded_injury_features)

print("\n" + "=" * 70)
print("✅ VERIFICATION COMPLETED SUCCESSFULLY!")
print("=" * 70)

In [ ]:
# ============================================
# STEP 19 — TEST INJURY PREDICTION MODEL
# ============================================

import joblib
import pandas as pd

# Load saved model and features
model = joblib.load(
    "/kaggle/working/injury_prediction_model/injury_prediction_model.pkl"
)

features = joblib.load(
    "/kaggle/working/injury_prediction_model/injury_prediction_features.pkl"
)

# --------------------------------------------
# Create a realistic safety observation
# --------------------------------------------

new_observation = {
    "Division": "Engineering & Project",
    "Observation Related To": "Employee",
    "Primary Cause": "Material Handling",
    "Working Condition": "Group Working",
    "Machine Condition": "Working",
    "Observation Type": "Unsafe Act & Unsafe Condition",
    "Incident Type": "Process"
}

# Convert observation into one-hot encoded format
input_df = pd.DataFrame([new_observation])

input_encoded = pd.get_dummies(
    input_df,
    drop_first=False,
    dtype=int
)

# Ensure EXACT same features and order as training
input_encoded = input_encoded.reindex(
    columns=features,
    fill_value=0
)

# --------------------------------------------
# Prediction
# --------------------------------------------

prediction = model.predict(input_encoded)[0]

probabilities = model.predict_proba(input_encoded)[0]

probability_dict = dict(
    zip(model.classes_, probabilities)
)

# --------------------------------------------
# Display result
# --------------------------------------------

print("=" * 75)
print("INJURY PREDICTION MODEL — TEST RESULT")
print("=" * 75)

print("\nInput Observation:")
for key, value in new_observation.items():
    print(f"{key}: {value}")

print("\nPredicted Injury Risk:")
print(prediction)

print("\nInjury Risk Probabilities:")

for injury_class, probability in sorted(
    probability_dict.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{injury_class}: {probability * 100:.2f}%")

print("\n" + "=" * 75)
print("✅ Injury prediction completed successfully!")
print("=" * 75)

In [ ]:
# ============================================
# STEP 20 — CREATE SAFETY AGENT BACKEND PACKAGE
# ============================================

import os
import shutil
import json

# --------------------------------------------
# Create package directory
# --------------------------------------------

package_dir = "/kaggle/working/Safety_Agent_Model_v1.0"

if os.path.exists(package_dir):
    shutil.rmtree(package_dir)

os.makedirs(package_dir)

# --------------------------------------------
# Copy Safety Risk Model
# --------------------------------------------

safety_source = "/kaggle/working/safety_risk_model"

safety_dest = os.path.join(
    package_dir,
    "safety_risk_model"
)

shutil.copytree(
    safety_source,
    safety_dest
)

# --------------------------------------------
# Copy Injury Prediction Model
# --------------------------------------------

injury_source = "/kaggle/working/injury_prediction_model"

injury_dest = os.path.join(
    package_dir,
    "injury_prediction_model"
)

shutil.copytree(
    injury_source,
    injury_dest
)

# --------------------------------------------
# Create package README
# --------------------------------------------

readme = """
SAFETY AGENT MODEL PACKAGE
==========================

Version:
Safety Agent Model v1.0

Purpose:
Construction Intelligence Hub — Safety Agent

COMPONENTS
----------

1. Safety Risk Model
   Algorithm: Gradient Boosting Classifier
   Training Records: 1000
   Features: 5
   Target: Risk_Level
   Classes: High, Low, Medium
   5-Fold CV Accuracy: 71.00%

2. Injury Prediction Model
   Algorithm: Logistic Regression
   Training Records: 950
   Features: 29
   Target: Injury
   Classes: 9 injury-risk categories
   3-Fold CV Accuracy: 62.11%

FILES
-----

safety_risk_model/
    safety_risk_model.pkl
    safety_risk_features.pkl
    metadata.json

injury_prediction_model/
    injury_prediction_model.pkl
    injury_prediction_features.pkl
    metadata.json

IMPORTANT
---------

The feature files must be used to maintain the exact feature
ordering expected by each trained model.

The injury model uses one-hot encoding.

Model outputs should be treated as predictions/probabilities,
not guaranteed outcomes.

Generated for:
Construction Intelligence Hub
"""

with open(
    os.path.join(package_dir, "README.txt"),
    "w"
) as f:
    f.write(readme.strip())

# --------------------------------------------
# Create ZIP
# --------------------------------------------

zip_path = "/kaggle/working/Safety_Agent_Model_v1.0"

shutil.make_archive(
    zip_path,
    "zip",
    package_dir
)

# --------------------------------------------
# Display package
# --------------------------------------------

print("=" * 75)
print("SAFETY AGENT MODEL PACKAGE")
print("=" * 75)

print("\n✅ Package created successfully!")

print("\nPackage contents:")

for root, dirs, files in os.walk(package_dir):
    level = root.replace(package_dir, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    └── {file}")

print("\nZIP file:")
print(zip_path + ".zip")

print("\n" + "=" * 75)
print("✅ SAFETY AGENT MODEL v1.0 READY")
print("=" * 75)

In [ ]:
# ============================================
# STEP 21 — CREATE FINAL SAFETY AGENT ZIP
# ============================================

import os
import shutil

# Source folders
safety_source = "/kaggle/working/safety_risk_model"
injury_source = "/kaggle/working/injury_prediction_model"

# Final package folder
package_dir = "/kaggle/working/Safety_Agent_Model_v1.0"

# Remove old package if it exists
if os.path.exists(package_dir):
    shutil.rmtree(package_dir)

# Create folders
os.makedirs(package_dir)
os.makedirs(os.path.join(package_dir, "safety_risk_model"))
os.makedirs(os.path.join(package_dir, "injury_prediction_model"))

# Copy Safety Risk model files
for file in os.listdir(safety_source):
    shutil.copy2(
        os.path.join(safety_source, file),
        os.path.join(package_dir, "safety_risk_model", file)
    )

# Copy Injury Prediction model files
for file in os.listdir(injury_source):
    shutil.copy2(
        os.path.join(injury_source, file),
        os.path.join(package_dir, "injury_prediction_model", file)
    )

# Copy README if it already exists
readme_path = os.path.join(
    "/kaggle/working",
    "Safety_Agent_Model_v1.0",
    "README.txt"
)

if os.path.exists(readme_path):
    shutil.copy2(
        readme_path,
        os.path.join(package_dir, "README.txt")
    )

# Create final ZIP
zip_base = "/kaggle/working/Safety_Agent_Model_v1.0_FINAL"

if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")

shutil.make_archive(
    zip_base,
    "zip",
    package_dir
)

# --------------------------------------------
# Verify ZIP
# --------------------------------------------

final_zip = zip_base + ".zip"

print("=" * 75)
print("FINAL SAFETY AGENT MODEL PACKAGE")
print("=" * 75)

print("\n✅ ZIP created successfully!")

print("\nZIP location:")
print(final_zip)

print("\nPackage contents:")

for root, dirs, files in os.walk(package_dir):
    level = root.replace(package_dir, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    └── {file}")

print("\n" + "=" * 75)
print("✅ FINAL ZIP READY FOR BACKEND TEAM")
print("=" * 75)